In [3]:
from pyspark.sql import SparkSession

DEFAULT_SPARK_REMOTE = "sc://localhost:15002"
spark = SparkSession.builder.remote(DEFAULT_SPARK_REMOTE).getOrCreate()

spark.version

'4.1.2'

In [5]:
from pyspark.sql.functions import (split, 
                                  to_json, 
                                  from_json, 
                                  lit,
                                  size,
                                  create_map
                                  )

df = spark.createDataFrame([[1, "This is a simple simple string"],
                              [2, "Hello World!"],], 
                             ["id", "Text"])

df = df.select(
    "id",
    "Text",
    to_json(
        create_map(lit("Words count"), size(split("Text", " ")))
    ).alias("WordsCount")
)

display(df)


DataFrame[id: bigint, Text: string, WordsCount: string]

In [7]:
from pyspark.sql.functions import schema_of_json
from pyspark.sql.functions import from_json, col


df.printSchema()

json_string = df.select("WordsCount").collect()[0][0]
json_schema = schema_of_json(json_string)

df2 = (df.withColumn("WordsCount",from_json("WordsCount", json_schema))
         .select("id", "Text", "WordsCount.*"))

df2.show()

root
 |-- id: long (nullable = true)
 |-- Text: string (nullable = true)
 |-- WordsCount: string (nullable = true)

+---+--------------------+-----------+
| id|                Text|Words count|
+---+--------------------+-----------+
|  1|This is a simple ...|          6|
|  2|        Hello World!|          2|
+---+--------------------+-----------+

